<a href="https://colab.research.google.com/github/ColdstreamerDawn/European-Power-Markets-/blob/main/Main%20Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Install the relevant libraries**

In [ ]:
!pip install entsoe-py

**Import the data from** **ENTSOE**

In [31]:
from entsoe import EntsoePandasClient
import pandas as pd
from itertools import permutations
from entsoe.exceptions import NoMatchingDataError
from xgboost import XGBRegressor

#Initialisation of data request, uses personal api key
client = EntsoePandasClient(api_key="5de489ac-9680-449e-a85d-3db36070c128")
start = pd.Timestamp('20200601', tz='UTC') #start date of the downloaded set
end = pd.Timestamp('20260602', tz='UTC') #end date of the downloaded set
country_code = 'PT'
neighbours = ["ES","PT"]
other_tech = ["B02", "B04", "B05", "B06", "B10", "B11", "B12", "B14"]

# 0.0.0 Create function to take convert to quarter hourly UTC timestamp
def make_quarter_hourly(df, ts_col="timestamp"):
  df = df.copy()
  df[ts_col] = pd.to_datetime(df[ts_col], utc=True)

  # Set timestamp as index
  df = df.set_index(ts_col).sort_index()

  # Keep only numeric columns
  numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
  df = df[numeric_cols]

  # Convert to 15-minute frequency
  # Hourly values are forward-filled into the four quarter-hours
  df = df.resample("15min").ffill()

  df.index.name = ts_col
  return df

# 0.0.1 Create function to add "timestamp" to first column and set as index, then adjust to hourly
def add_timestamp(df):
  df = df.reset_index()
  df = df.rename(columns={"index": "timestamp"})
  df = make_quarter_hourly(df)
  return df

# 0.1.0 - RENEWABLES GENERATION FORECAST FUNCTION
dfs_RES =[]
def RES_forecast(country_code, startdate, enddate):

  #Make the request
  df = client.query_wind_and_solar_forecast(country_code, start=startdate, end=enddate, psr_type=None)
  df = add_timestamp(df)

  #Pick up any columns with "wind" in them and sum together for simplicity,then also add country code
  wind_cols = [c for c in df.columns if "wind" in c.lower()]
  df["Wind"] = df[wind_cols].sum(axis=1)
  df = df.drop(columns=wind_cols)
  df = df.rename(columns=lambda c: f"{country_code}_{c}")
  dfs_RES.append(df)

# 0.2.0 - POWER PRICE DATA FUNCTION
dfs_prices = []
def price_data(country_code, startdate, enddate):

  #Day-Ahead price request
  df_da = client.query_day_ahead_prices(country_code, start=startdate, end=enddate)
  df_da = add_timestamp(df_da)
  df_da = df_da.rename(columns={0: f"DA_price"})

  #Imbalance price request
  df_Imb = client.query_imbalance_prices(country_code, start=startdate, end=enddate)
  df_Imb = add_timestamp(df_Imb)
  df_Imb = df_Imb.rename(columns={"Long":  "Imb_long", "Short": "Imb_short"})

  #Concatenate DA and Imbalance prices
  df = pd.concat([df_da, df_Imb], axis=1)
  dfs_prices.append(df)

# 0.3.0 - CROSS-BORDER EXCHANGESS DATA FUNCTION
dfs_exchanges = []
def cross_border_exchanges(country_code_from, country_code_to, startdate, enddate):
  try:
    #Make the request, and name the column based on flows
    df = client.query_scheduled_exchanges(country_code_from, country_code_to, start=startdate, end=enddate)
    df = add_timestamp(df)
    df = df.rename(columns={0: f"{country_code_from}_{country_code_to}"})
    dfs_exchanges.append(df)

  except NoMatchingDataError:
    print(f"NoMatchingDataError: No data for cross-border exchanges from {country_code_from} to {country_code_to} in the specified period.")
    # If no data, append an empty DataFrame to avoid errors in pd.concat later
    # The column name is important for later concatenation and feature selection
    empty_df = pd.DataFrame(columns=[f"{country_code_from}_{country_code_to}"])
    # Set the index name to 'timestamp' for consistency with other dataframes after add_timestamp
    empty_df.index.name = 'timestamp'
    dfs_exchanges.append(empty_df)

# 0.3.1 CROSS-BORDER Permutations function, iterates through "neighbours" permutations
def cross_border_permutations(neighbours, startdate, enddate):
 flows = list(permutations(neighbours, 2))
 for flow in flows:
  cross_border_exchanges(flow[0], flow[1], startdate, enddate)

# 0.4.0 LOAD FORECAST FUNCTION
dfs_load = []
def load_forecast(country_code, startdate, enddate):

  #Make the request
  df = client.query_load_forecast(country_code, start=startdate, end=enddate)
  df = add_timestamp(df)
  df = df.rename(columns={"Forecasted Load": f"{country_code}_load"})
  dfs_load.append(df)

# 1.0.0 Fetch data
for countries in neighbours:
  RES_forecast(countries, start, end)
  load_forecast(countries, start, end)
cross_border_permutations(neighbours, start, end) #iterates over neighbours
price_data(country_code, start, end) #Only takes target country prices

# 1.0.1 merge all dataframes in dfs along the timestamp column, and make timestamp index column
df = pd.concat(dfs_exchanges + dfs_prices + dfs_load + dfs_RES, axis=1)
df = df.reset_index()
print(df.head(24))

                   timestamp  ES_PT  PT_ES  DA_price  Imb_long  Imb_short  \
0  2020-06-01 00:00:00+00:00  442.4    0.0     34.26     28.73      38.90   
1  2020-06-01 00:15:00+00:00  442.4    0.0     34.26     28.73      38.90   
2  2020-06-01 00:30:00+00:00  442.4    0.0     34.26     28.73      38.90   
3  2020-06-01 00:45:00+00:00  442.4    0.0     34.26     28.73      38.90   
4  2020-06-01 01:00:00+00:00  381.8    0.0     33.52     32.87      34.63   
5  2020-06-01 01:15:00+00:00  381.8    0.0     33.52     32.87      34.63   
6  2020-06-01 01:30:00+00:00  381.8    0.0     33.52     32.87      34.63   
7  2020-06-01 01:45:00+00:00  381.8    0.0     33.52     32.87      34.63   
8  2020-06-01 02:00:00+00:00  330.6    0.0     33.50     32.32      35.76   
9  2020-06-01 02:15:00+00:00  330.6    0.0     33.50     32.32      35.76   
10 2020-06-01 02:30:00+00:00  330.6    0.0     33.50     32.32      35.76   
11 2020-06-01 02:45:00+00:00  330.6    0.0     33.50     32.32      35.76   

**Define the Prediction Model for each Price feature DA/ImbLong/ImbShort**

In [42]:
#2.0.0 This function takes each market and trains/tests on it
def run_price_model(df, target_col, target_name, test_date, test_days=10):

  df_model = df.copy()
  df_model["power_price"] = df_model[target_col]
  old_lags = [c for c in df_model.columns if "lag" in c.lower()]
  df_model = df_model.drop(columns=old_lags, errors="ignore")

  #2.0.1 Regular time-based variables
  df_model["hour"] = df_model["timestamp"].dt.hour
  df_model["day_of_week"] = df_model["timestamp"].dt.dayofweek
  df_model["month"] = df_model["timestamp"].dt.month
  df_model["weekend"] = (df_model["timestamp"].dt.weekday >= 5).astype(int)


  #2.0.2 Logic such that DA forecasts don't take less than 24h price notice
  #because DA prices are set the day before
  if target_name == "DA":
    df_model[f"{target_name}_lag_24h"] = df_model["power_price"].shift(96)
    df_model[f"{target_name}_lag_48h"] = df_model["power_price"].shift(192)
    df_model[f"{target_name}_lag_72h"] = df_model["power_price"].shift(288)
    df_model[f"{target_name}_lag_1w"] = df_model["power_price"].shift(672)
  else:
    df_model[f"{target_name}_lag_15m"] = df_model["power_price"].shift(1)
    df_model[f"{target_name}_lag_1h"] = df_model["power_price"].shift(4)
    df_model[f"{target_name}_lag_24h"] = df_model["power_price"].shift(96)
    df_model[f"{target_name}_lag_1w"] = df_model["power_price"].shift(672)

  # 2.0.3 Calculates residual load in the dataframe
  for countries in neighbours:
    df_model[f"{countries}_load_residual"] = (df_model[f"{countries}_load"]
        - df_model[f"{countries}_Wind"] - df_model[f"{countries}_Solar"])

  #2.0.4 Drop any columns that are entirely NaN and picked up as no matching data for cross border exchanges
  df_model = df_model.dropna(axis=1, how="all")

  #2.0.5 Note that we are testing over a set number of days defined in the function
  test_end = test_date + pd.Timedelta(days=test_days)

  #2.0.6 The splitting of the training and testing datasets
  train = df_model[df_model["timestamp"] < test_date].copy()
  test = df_model[(df_model["timestamp"] >= test_date)
      & (df_model["timestamp"] < test_end)].copy()

  drop_cols = ["timestamp", "power_price", target_col]
  other_targets = ["DA_price", "Imb_long", "Imb_short"]
  drop_cols += [c for c in other_targets if c in df_model.columns and c != target_col]

  X_train = train.drop(columns=drop_cols, errors="ignore")
  X_test = test.drop(columns=drop_cols, errors="ignore")
  Y_train = train["power_price"]
  Y_test = test["power_price"]

  #2.0.7 The training of the XGBoost model
  model = XGBRegressor(
      learning_rate=0.05,
      max_depth=5,
      min_child_weight=1,
      n_estimators=1000,
      n_jobs=-1,
      objective="reg:squarederror",
      tree_method="hist",
      random_state=42)
  model.fit(X_train, Y_train)

  #2.0.8 Testing the model
  Y_pred = model.predict(X_test)
  preds = pd.DataFrame({"timestamp": test["timestamp"],f"{target_name}_forecast": Y_pred})
  combined = pd.merge(preds, test[["timestamp", "power_price"]], on="timestamp").dropna(subset=["power_price"])

  #2.0.9 Model metrics
  rmse_value = np.sqrt(mean_squared_error(combined["power_price"],combined[f"{target_name}_forecast"]))

  print(target_name)
  print("X_train shape:", X_train.shape)
  print("Y_train shape:", Y_train.shape)
  print("MSE:", mean_squared_error(combined["power_price"], combined[f"{target_name}_forecast"]))
  print("RMSE:", rmse_value)
  print("MAE:", mean_absolute_error(combined["power_price"], combined[f"{target_name}_forecast"]))
  print("R2:", r2_score(combined["power_price"], combined[f"{target_name}_forecast"]))

  return model, preds, combined, X_test


**Running the model**

In [43]:
#3.0.0 Input the test date
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
test_date = pd.Timestamp("2026-05-03", tz="UTC")

#3.0.1 The Day-Ahead price model
model_DA, preds_DA, combined_DA, X_test_DA = run_price_model(df, "DA_price", "DA", test_date)

#3.0.2 The Long Imbalance model
model_long, preds_long, combined_long, X_test_long = run_price_model(df, "Imb_long", "Long", test_date)

#3.0.3 The Short Imbalance model
model_short, preds_short, combined_short, X_test_short = run_price_model(df, "Imb_short", "Short", test_date)

DA
X_train shape: (207552, 18)
Y_train shape: (207552,)
MSE: 233.26776846447058
RMSE: 15.273106051634375
MAE: 11.04976646810615
R2: 0.8989576197139837
Long
X_train shape: (207552, 18)
Y_train shape: (207552,)
MSE: 1347.1511333296196
RMSE: 36.7035575023678
MAE: 17.965902345265242
R2: 0.5547462645127861
Short
X_train shape: (207552, 18)
Y_train shape: (207552,)
MSE: 1336.7067529028157
RMSE: 36.561000436295714
MAE: 19.195827930465338
R2: 0.5977246754529353


**Charts**

In [ ]:
#4.0.0 The Charts
import matplotlib.pyplot as plt

plot_sets = [
    ("Day-Ahead Price", combined_DA, "DA_forecast"),
    ("Imbalance Long Price", combined_long, "Long_forecast"),
    ("Imbalance Short Price", combined_short, "Short_forecast"),]

for title, data, forecast_col in plot_sets:
    plt.figure(figsize=(15, 5))
    plt.plot(data["timestamp"], data["power_price"], label="Actual")
    plt.plot(data["timestamp"], data[forecast_col], label="Forecast")

    plt.title(title)
    plt.xlabel("Timestamp")
    plt.ylabel("€/MWh")
    plt.legend()
    plt.show()

**Trading algorithm**

In [44]:
strategy_rows = []

for day in combined_DA["timestamp"].dt.date.unique():

    da_day = combined_DA[combined_DA["timestamp"].dt.date == day].copy()
    long_day = combined_long[combined_long["timestamp"].dt.date == day].copy()
    short_day = combined_short[combined_short["timestamp"].dt.date == day].copy()

    pred_max_idx = da_day["DA_forecast"].idxmax()
    pred_min_idx = da_day["DA_forecast"].idxmin()

    max_timestamp = da_day.loc[pred_max_idx, "timestamp"]
    min_timestamp = da_day.loc[pred_min_idx, "timestamp"]

    pred_DA_max = da_day.loc[pred_max_idx, "DA_forecast"]
    actual_DA_at_max = da_day.loc[pred_max_idx, "power_price"]

    pred_DA_min = da_day.loc[pred_min_idx, "DA_forecast"]
    actual_DA_at_min = da_day.loc[pred_min_idx, "power_price"]

    long_max_row = long_day[long_day["timestamp"] == max_timestamp].iloc[0]
    short_max_row = short_day[short_day["timestamp"] == max_timestamp].iloc[0]

    long_min_row = long_day[long_day["timestamp"] == min_timestamp].iloc[0]
    short_min_row = short_day[short_day["timestamp"] == min_timestamp].iloc[0]

    pred_charge_imb_price = long_min_row["Long_forecast"]
    actual_charge_imb_price = long_min_row["power_price"]

    pred_discharge_imb_price = short_max_row["Short_forecast"]
    actual_discharge_imb_price = short_max_row["power_price"]

    pred_charge_spread = pred_charge_imb_price - pred_DA_min
    actual_charge_spread = actual_charge_imb_price - actual_DA_at_min

    pred_discharge_spread = pred_discharge_imb_price - pred_DA_max
    actual_discharge_spread = actual_discharge_imb_price - actual_DA_at_max

    charge_market_decision = "Imbalance" if pred_charge_imb_price < pred_DA_min else "DA"
    discharge_market_decision = "Imbalance" if pred_discharge_imb_price > pred_DA_max else "DA"

    realised_charge_value = (
        actual_charge_imb_price
        if charge_market_decision == "Imbalance"
        else actual_DA_at_min
    )

    realised_discharge_value = (
        actual_discharge_imb_price
        if discharge_market_decision == "Imbalance"
        else actual_DA_at_max
    )

    actual_best_charge_market = (
        "Imbalance"
        if actual_charge_imb_price < actual_DA_at_min
        else "DA"
    )

    actual_best_discharge_market = (
        "Imbalance"
        if actual_discharge_imb_price > actual_DA_at_max
        else "DA"
    )

    perfect_charge_value = min(actual_DA_at_min, actual_charge_imb_price)
    perfect_discharge_value = max(actual_DA_at_max, actual_discharge_imb_price)

    realised_battery_spread = realised_discharge_value - realised_charge_value
    perfect_battery_spread = perfect_discharge_value - perfect_charge_value
    regret = perfect_battery_spread - realised_battery_spread

    actual_DA_day_min = da_day["power_price"].min()
    actual_DA_day_max = da_day["power_price"].max()

    actual_DA_day_min_timestamp = da_day.loc[da_day["power_price"].idxmin(), "timestamp"]
    actual_DA_day_max_timestamp = da_day.loc[da_day["power_price"].idxmax(), "timestamp"]

    perfect_DA_only_spread = actual_DA_day_max - actual_DA_day_min

    actual_charge_imb_day_min = long_day["power_price"].min()
    actual_discharge_imb_day_max = short_day["power_price"].max()

    perfect_all_markets_charge_value = min(actual_DA_day_min, actual_charge_imb_day_min)
    perfect_all_markets_discharge_value = max(actual_DA_day_max, actual_discharge_imb_day_max)

    perfect_all_markets_spread = (
        perfect_all_markets_discharge_value
        - perfect_all_markets_charge_value
    )
    strategy_rows.append({
        "date": day,

        "PREDICTED": "",
        "Predicted Charge Hour": min_timestamp.hour,
        "Predicted Discharge Hour": max_timestamp.hour,
        "Predicted DA Charge Price": pred_DA_min,
        "Predicted DA Discharge Price": pred_DA_max,
        "Predicted Charge Imbalance Price": pred_charge_imb_price,
        "Predicted Discharge Imbalance Price": pred_discharge_imb_price,
        "Predicted Charge Spread": pred_charge_spread,
        "Predicted Discharge Spread": pred_discharge_spread,
        "Charge Market Decision": charge_market_decision,
        "Discharge Market Decision": discharge_market_decision,

        "ACTUAL": "",
        "Actual DA Charge Price": actual_DA_at_min,
        "Actual DA Discharge Price": actual_DA_at_max,
        "Actual Charge Imbalance Price": actual_charge_imb_price,
        "Actual Discharge Imbalance Price": actual_discharge_imb_price,
        "Actual Charge Spread": actual_charge_spread,
        "Actual Discharge Spread": actual_discharge_spread,
        "Actual Best Charge Market": actual_best_charge_market,
        "Actual Best Discharge Market": actual_best_discharge_market,

        "OUTCOME": "",
        "Realised Charge Value": realised_charge_value,
        "Realised Discharge Value": realised_discharge_value,
        "Realised Battery Spread": realised_battery_spread,
        "Perfect Foresight Charge Value": perfect_charge_value,
        "Perfect Foresight Discharge Value": perfect_discharge_value,
        "Perfect Foresight Battery Spread": perfect_battery_spread,
        "Regret": regret,
        "Perfect DA-Only Charge Value": actual_DA_day_min,
        "Perfect DA-Only Charge Hour": actual_DA_day_min_timestamp.hour,
        "Perfect DA-Only Discharge Value": actual_DA_day_max,
        "Perfect DA-Only Discharge Hour": actual_DA_day_max_timestamp.hour,
        "Perfect DA-Only Battery Spread": perfect_DA_only_spread,

        "Perfect All-Markets Charge Value": perfect_all_markets_charge_value,
        "Perfect All-Markets Discharge Value": perfect_all_markets_discharge_value,
        "Perfect All-Markets Battery Spread": perfect_all_markets_spread,
    })

strategy_df = pd.DataFrame(strategy_rows)

strategy_df_flipped = strategy_df.set_index("date").T

print(strategy_df_flipped)

date                                 2026-05-03  2026-05-04  2026-05-05  \
PREDICTED                                                                 
Predicted Charge Hour                         8          14          14   
Predicted Discharge Hour                     20          19           5   
Predicted DA Charge Price             -7.649681   -0.488309   -2.148991   
Predicted DA Discharge Price         104.069466  114.965195  126.933159   
Predicted Charge Imbalance Price      34.463562  -10.554866  -14.809688   
Predicted Discharge Imbalance Price  122.280777     79.2929  139.507355   
Predicted Charge Spread               42.113243  -10.066557  -12.660696   
Predicted Discharge Spread            18.211311  -35.672295   12.574196   
Charge Market Decision                       DA   Imbalance   Imbalance   
Discharge Market Decision             Imbalance          DA   Imbalance   
ACTUAL                                                                    
Actual DA Charge Price   

In [49]:
battery_capacity_mwh = 1.0
battery_power_mw = 1.0
interval_hours = 0.25
energy_per_interval = battery_power_mw * interval_hours

soc = 0.0
strategy_rows = []
dispatch_rows = []

for day in combined_DA["timestamp"].dt.date.unique():

    day_start_soc = soc

    da_day = combined_DA[combined_DA["timestamp"].dt.date == day].copy()
    long_day = combined_long[combined_long["timestamp"].dt.date == day].copy()
    short_day = combined_short[combined_short["timestamp"].dt.date == day].copy()

    day_df = (
        da_day[["timestamp", "DA_forecast", "power_price"]]
        .rename(columns={"power_price": "DA_actual"})
        .merge(
            long_day[["timestamp", "Long_forecast", "power_price"]]
            .rename(columns={
                "Long_forecast": "imb_charge_forecast",
                "power_price": "imb_charge_actual"
            }),
            on="timestamp",
            how="left"
        )
        .merge(
            short_day[["timestamp", "Short_forecast", "power_price"]]
            .rename(columns={
                "Short_forecast": "imb_discharge_forecast",
                "power_price": "imb_discharge_actual"
            }),
            on="timestamp",
            how="left"
        )
        .sort_values("timestamp")
        .reset_index(drop=True)
    )

    charge_candidates = day_df.nsmallest(4, "DA_forecast").copy()
    charge_candidates["action"] = "charge"

    discharge_candidates = day_df.nlargest(4, "DA_forecast").copy()
    discharge_candidates["action"] = "discharge"

    candidates = (
        pd.concat([charge_candidates, discharge_candidates])
        .sort_values("timestamp")
        .reset_index(drop=True)
    )

    daily_cashflow = 0.0

    for _, row in candidates.iterrows():

        timestamp = row["timestamp"]
        action = row["action"]

        if action == "charge" and soc < battery_capacity_mwh:

            energy = min(energy_per_interval, battery_capacity_mwh - soc)

            if row["imb_charge_forecast"] < row["DA_forecast"]:
                market = "Imbalance"
                predicted_price = row["imb_charge_forecast"]
                actual_price = row["imb_charge_actual"]
            else:
                market = "DA"
                predicted_price = row["DA_forecast"]
                actual_price = row["DA_actual"]

            cashflow = -energy * actual_price
            soc += energy

        elif action == "discharge" and soc > 0:

            energy = min(energy_per_interval, soc)

            if row["imb_discharge_forecast"] > row["DA_forecast"]:
                market = "Imbalance"
                predicted_price = row["imb_discharge_forecast"]
                actual_price = row["imb_discharge_actual"]
            else:
                market = "DA"
                predicted_price = row["DA_forecast"]
                actual_price = row["DA_actual"]

            cashflow = energy * actual_price
            soc -= energy

        else:
            continue

        daily_cashflow += cashflow

        dispatch_rows.append({
            "date": day,
            "timestamp": timestamp,
            "action": action,
            "market": market,
            "energy_mwh": energy,
            "soc_after": soc,
            "DA_forecast": row["DA_forecast"],
            "DA_actual": row["DA_actual"],
            "imb_charge_forecast": row["imb_charge_forecast"],
            "imb_charge_actual": row["imb_charge_actual"],
            "imb_discharge_forecast": row["imb_discharge_forecast"],
            "imb_discharge_actual": row["imb_discharge_actual"],
            "selected_predicted_price": predicted_price,
            "selected_actual_price": actual_price,
            "cashflow": cashflow
        })

    dispatch_day = pd.DataFrame([r for r in dispatch_rows if r["date"] == day])

    total_charge_cost = -dispatch_day.loc[
        dispatch_day["action"] == "charge", "cashflow"
    ].sum()

    total_discharge_revenue = dispatch_day.loc[
        dispatch_day["action"] == "discharge", "cashflow"
    ].sum()

    realised_spread = (
        total_discharge_revenue - total_charge_cost
    )

    strategy_rows.append({
        "date": day,
        "charge_intervals": (dispatch_day["action"] == "charge").sum(),
        "discharge_intervals": (dispatch_day["action"] == "discharge").sum(),
        "total_charge_cost": total_charge_cost,
        "total_discharge_revenue": total_discharge_revenue,
        "realised_spread": realised_spread,
        "daily_cashflow": daily_cashflow,
        "start_soc": day_start_soc,
        "final_soc": soc
    })

dispatch_df = pd.DataFrame(dispatch_rows)
strategy_df = pd.DataFrame(strategy_rows)

print("DISPATCH TABLE")
print(dispatch_df)

print("DAILY STRATEGY SUMMARY")
print(strategy_df)
dispatch_df.to_csv("battery_strategy_summary.csv", index=False)

from google.colab import files
files.download("battery_strategy_summary.csv")

DISPATCH TABLE
          date                 timestamp     action     market  energy_mwh  \
0   2026-05-03 2026-05-03 08:45:00+00:00     charge         DA        0.25   
1   2026-05-03 2026-05-03 09:30:00+00:00     charge         DA        0.25   
2   2026-05-03 2026-05-03 09:45:00+00:00     charge         DA        0.25   
3   2026-05-03 2026-05-03 10:45:00+00:00     charge         DA        0.25   
4   2026-05-03 2026-05-03 20:00:00+00:00  discharge  Imbalance        0.25   
..         ...                       ...        ...        ...         ...   
67  2026-05-12 2026-05-12 11:45:00+00:00     charge         DA        0.25   
68  2026-05-12 2026-05-12 19:15:00+00:00  discharge         DA        0.25   
69  2026-05-12 2026-05-12 19:30:00+00:00  discharge         DA        0.25   
70  2026-05-12 2026-05-12 19:45:00+00:00  discharge         DA        0.25   
71  2026-05-12 2026-05-12 20:00:00+00:00  discharge         DA        0.25   

    soc_after  DA_forecast  DA_actual  imb_charg

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>